In [2]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI

In [3]:
load_dotenv()  # .env 파일을 환경변수로 등록

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', "neo4j")  # 기본값 neo4j

In [4]:
# Neo4jGraph : Neo4j 연결정보를 받아서, 그래프 조화와 스키마 확인 기능을 하는 래퍼 클래스
graph = Neo4jGraph(
    url = NEO4J_URI,
    username = NEO4J_USERNAME,
    password = NEO4J_PASSWORD,
    database = NEO4J_DATABASE
)

print("Langchain과 Neo4j 연결 성공")

Langchain과 Neo4j 연결 성공


In [ ]:
graph.refresh_schema()  # 스키마 새로고침 (그래프 구조 변경시 실행)

print(graph.schema)  # 노드 레이블, 속성, 관계 유형과 방향

Node properties:
Student {age: INTEGER, name: STRING, student_id: INTEGER}
Course {name: STRING, course_id: INTEGER, level: STRING, duration: INTEGER}
Instructor {name: STRING, career: INTEGER, instructor_id: INTEGER}
Category {name: STRING, category_id: INTEGER}
Relationship properties:
ENROLLED_IN {score: INTEGER, enrolled_at: DATE}
The relationships:
(:Student)-[:ENROLLED_IN]->(:Course)
(:Course)-[:BELONGS_TO]->(:Category)
(:Instructor)-[:TEACHES]->(:Course)


In [ ]:
query = """
MATCH (student:Student)-[:ENROLLED_IN]->(course:Course)
RETURN
    student.name AS student_name,
    course.name AS course_name,
    student.student_id AS student_id,
    course.course_id AS course_id
ORDER BY student_id, course_id;
"""

result = graph.query(query)  # 쿼리를 받아 결과를 list(dict)로 반환

result

[{'student_name': '홍길동',
  'course_name': 'Python',
  'student_id': 1,
  'course_id': 101},
 {'student_name': '홍길동',
  'course_name': 'Data a',
  'student_id': 1,
  'course_id': 104},
 {'student_name': '김영희',
  'course_name': '백이번',
  'student_id': 2,
  'course_id': 102},
 {'student_name': '김영희',
  'course_name': '백삼',
  'student_id': 2,
  'course_id': 103},
 {'student_name': '이민수',
  'course_name': 'Python',
  'student_id': 3,
  'course_id': 101},
 {'student_name': '이민수',
  'course_name': 'Data a',
  'student_id': 3,
  'course_id': 104},
 {'student_name': '박서연',
  'course_name': '백삼',
  'student_id': 4,
  'course_id': 103},
 {'student_name': '박서연',
  'course_name': '백오',
  'student_id': 4,
  'course_id': 105},
 {'student_name': '최준호',
  'course_name': '백이번',
  'student_id': 5,
  'course_id': 102},
 {'student_name': '최준호',
  'course_name': '백육',
  'student_id': 5,
  'course_id': 106}]

In [7]:
llm = ChatOpenAI(
    model = os.getenv('OPENAI_MODEL'),
    temperature=0  # DB의 정보만 가져올 것이므로 창의성은 0 (결정론적인 답변 = 일관적)
)

In [8]:
# LLM과 Neo4j를 연결하여 자연어 질문을 Cypher로 변환하고 답변하는 체인
chain = GraphCypherQAChain.from_llm(
    llm = llm,  # Cypher 생성 및 최종 답변 llm
    graph = graph,  # 참조할 Neo4j 그래프 객체
    verbose = True,  # 로그 출력
    validate_cypher = True,  # 생성된 Cypher 검증
    return_intermediate_steps = True,  # 중간과정 함께 반환
    top_k = 10,  # 조회결과 10개
    allow_dangerous_requests = True  # DB 쿼리 실행 위험성 확인
)

In [9]:
question = "Python 강의를 수강하는 학생을 알려줘."

response = chain.invoke({"query": question})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
WHERE c.name = 'Python'
RETURN s;

Full Context:
[{'s': {'name': '홍길동', 'student_id': 1, 'age': 26}}, {'s': {'name': '이민수', 'student_id': 3, 'age': 24}}]

> Finished chain.


{'query': 'Python 강의를 수강하는 학생을 알려줘.',
 'result': 'Python 강의를 수강하는 학생은 홍길동과 이민수입니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nWHERE c.name = 'Python'\nRETURN s;\n"},
  {'context': [{'s': {'name': '홍길동', 'student_id': 1, 'age': 26}},
    {'s': {'name': '이민수', 'student_id': 3, 'age': 24}}]}]}

In [10]:
response['result']  # 최종 답변만 확인

'Python 강의를 수강하는 학생은 홍길동과 이민수입니다.'

In [11]:
response['intermediate_steps']  # 중간 과정만 확인

[{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nWHERE c.name = 'Python'\nRETURN s;\n"},
 {'context': [{'s': {'name': '홍길동', 'student_id': 1, 'age': 26}},
   {'s': {'name': '이민수', 'student_id': 3, 'age': 24}}]}]

In [12]:
# 질문 답변 chain 함수
def ask_graph(question: str) -> dict:
    if not question.strip():
        raise ValueError('질문을 입력하셔야 합니다')

    response = chain.invoke({'query': question})

    print(f'[질문] {question}')
    print(f'[최종 답변] {response['result']}')

    # 중간과정 추가시
    for step in response.get('intermediate_steps', []):
        if 'query' in step:
            print(f'[생성된 Cypher] {step['query']}')
        if 'context' in step:
            print(f'[조회 결과] {step['context']}')

    return response

In [15]:
ask_graph('홍길동이 수강하는 강의와, 해당 강의의 담당 강사를 알려줘')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN c.name AS course_name, i.name AS instructor_name
Full Context:
[{'course_name': 'Python', 'instructor_name': 'Capybara'}, {'course_name': 'Data Analysis', 'instructor_name': 'Alice'}]

> Finished chain.
[질문] 홍길동이 수강하는 강의와, 해당 강의의 담당 강사를 알려줘
[최종 답변] 홍길동이 수강하는 강의와 담당 강사에 대한 정보는 알 수 없습니다.
[생성된 Cypher] MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN c.name AS course_name, i.name AS instructor_name
[조회 결과] [{'course_name': 'Python', 'instructor_name': 'Capybara'}, {'course_name': 'Data Analysis', 'instructor_name': 'Alice'}]


{'query': '홍길동이 수강하는 강의와, 해당 강의의 담당 강사를 알려줘',
 'result': '홍길동이 수강하는 강의와 담당 강사에 대한 정보는 알 수 없습니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)\nRETURN c.name AS course_name, i.name AS instructor_name"},
  {'context': [{'course_name': 'Python', 'instructor_name': 'Capybara'},
    {'course_name': 'Data Analysis', 'instructor_name': 'Alice'}]}]}

In [16]:
ask_graph('Tell me about the lectures 홍길동 took and the instructor in charge')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student {name: '홍길동'})-[e:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN c.name AS course_name,
       c.course_id AS course_id,
       c.level AS level,
       c.duration AS duration,
       e.score AS score,
       e.enrolled_at AS enrolled_at,
       i.name AS instructor_name,
       i.instructor_id AS instructor_id,
       i.career AS instructor_career;
Full Context:
[{'course_name': 'Python', 'course_id': 101, 'level': '초급', 'duration': 40, 'score': 95, 'enrolled_at': neo4j.time.Date(2026, 8, 1), 'instructor_name': 'Capybara', 'instructor_id': 1, 'instructor_career': 3}, {'course_name': 'Data Analysis', 'course_id': 104, 'level': '입문', 'duration': 40, 'score': 90, 'enrolled_at': neo4j.time.Date(2026, 8, 3), 'instructor_name': 'Alice', 'instructor_id': 2, 'instructor_career': 7}]

> Finished chain.
[질문] Tell me about the lectures 홍길동 took and the instructor in charge
[최종 답변] 홍길동은 다음 강의를 수강했습니다.

- *

{'query': 'Tell me about the lectures 홍길동 took and the instructor in charge',
 'result': '홍길동은 다음 강의를 수강했습니다.\n\n- **Python**: 초급, 40시간, 점수 95점, 2026년 8월 1일 수강 시작  \n  - 담당 강사: **Capybara** (경력 3년)\n- **Data Analysis**: 입문, 40시간, 점수 90점, 2026년 8월 3일 수강 시작  \n  - 담당 강사: **Alice** (경력 7년)',
 'intermediate_steps': [{'query': "MATCH (s:Student {name: '홍길동'})-[e:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)\nRETURN c.name AS course_name,\n       c.course_id AS course_id,\n       c.level AS level,\n       c.duration AS duration,\n       e.score AS score,\n       e.enrolled_at AS enrolled_at,\n       i.name AS instructor_name,\n       i.instructor_id AS instructor_id,\n       i.career AS instructor_career;"},
  {'context': [{'course_name': 'Python',
     'course_id': 101,
     'level': '초급',
     'duration': 40,
     'score': 95,
     'enrolled_at': neo4j.time.Date(2026, 8, 1),
     'instructor_name': 'Capybara',
     'instructor_id': 1,
     'instructor_career': 3},
    {'course_name':

In [17]:
ask_graph('인공지능 카테고리에 속한 강의를 알려줘')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Course)-[:BELONGS_TO]->(cat:Category {name: '인공지능'})
RETURN c.name AS course_name, c.course_id AS course_id, c.level AS level, c.duration AS duration;
Full Context:
[{'course_name': 'Machine Learning', 'course_id': 103, 'level': '입문', 'duration': 48}, {'course_name': 'Deep Learning', 'course_id': 105, 'level': '중급', 'duration': 52}, {'course_name': 'Langchain', 'course_id': 106, 'level': '중급', 'duration': 32}]

> Finished chain.
[질문] 인공지능 카테고리에 속한 강의를 알려줘
[최종 답변] 인공지능 카테고리에 속한 강의는 다음과 같습니다:

- Machine Learning (입문, 48시간)
- Deep Learning (중급, 52시간)
- Langchain (중급, 32시간)
[생성된 Cypher] MATCH (c:Course)-[:BELONGS_TO]->(cat:Category {name: '인공지능'})
RETURN c.name AS course_name, c.course_id AS course_id, c.level AS level, c.duration AS duration;
[조회 결과] [{'course_name': 'Machine Learning', 'course_id': 103, 'level': '입문', 'duration': 48}, {'course_name': 'Deep Learning', 'course_id': 105, 'level': '중급', 'duration': 52}, 

{'query': '인공지능 카테고리에 속한 강의를 알려줘',
 'result': '인공지능 카테고리에 속한 강의는 다음과 같습니다:\n\n- Machine Learning (입문, 48시간)\n- Deep Learning (중급, 52시간)\n- Langchain (중급, 32시간)',
 'intermediate_steps': [{'query': "MATCH (c:Course)-[:BELONGS_TO]->(cat:Category {name: '인공지능'})\nRETURN c.name AS course_name, c.course_id AS course_id, c.level AS level, c.duration AS duration;"},
  {'context': [{'course_name': 'Machine Learning',
     'course_id': 103,
     'level': '입문',
     'duration': 48},
    {'course_name': 'Deep Learning',
     'course_id': 105,
     'level': '중급',
     'duration': 52},
    {'course_name': 'Langchain',
     'course_id': 106,
     'level': '중급',
     'duration': 32}]}]}

In [18]:
ask_graph('수강생이 가장 많은 강의를 알려줘')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
RETURN c.name AS course_name, c.course_id AS course_id, count(s) AS student_count
ORDER BY student_count DESC
LIMIT 1
Full Context:
[{'course_name': 'Python', 'course_id': 101, 'student_count': 2}]

> Finished chain.
[질문] 수강생이 가장 많은 강의를 알려줘
[최종 답변] 수강생이 가장 많은 강의는 Python이며, 수강생은 2명입니다.
[생성된 Cypher] MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
RETURN c.name AS course_name, c.course_id AS course_id, count(s) AS student_count
ORDER BY student_count DESC
LIMIT 1
[조회 결과] [{'course_name': 'Python', 'course_id': 101, 'student_count': 2}]


{'query': '수강생이 가장 많은 강의를 알려줘',
 'result': '수강생이 가장 많은 강의는 Python이며, 수강생은 2명입니다.',
 'intermediate_steps': [{'query': 'MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nRETURN c.name AS course_name, c.course_id AS course_id, count(s) AS student_count\nORDER BY student_count DESC\nLIMIT 1'},
  {'context': [{'course_name': 'Python',
     'course_id': 101,
     'student_count': 2}]}]}

In [19]:
ask_graph('홍길동과 같은 강의를 수강한 다른 학생을 알려줘')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (hong:Student {name: '홍길동'})-[:ENROLLED_IN]->(course:Course)<-[:ENROLLED_IN]-(other:Student)
WHERE other.student_id <> hong.student_id
RETURN DISTINCT other.name AS student_name;

Full Context:
[{'student_name': '이민수'}]

> Finished chain.
[질문] 홍길동과 같은 강의를 수강한 다른 학생을 알려줘
[최종 답변] 홍길동과 같은 강의를 수강한 다른 학생은 알 수 없습니다.
[생성된 Cypher] MATCH (hong:Student {name: '홍길동'})-[:ENROLLED_IN]->(course:Course)<-[:ENROLLED_IN]-(other:Student)
WHERE other.student_id <> hong.student_id
RETURN DISTINCT other.name AS student_name;

[조회 결과] [{'student_name': '이민수'}]


{'query': '홍길동과 같은 강의를 수강한 다른 학생을 알려줘',
 'result': '홍길동과 같은 강의를 수강한 다른 학생은 알 수 없습니다.',
 'intermediate_steps': [{'query': "MATCH (hong:Student {name: '홍길동'})-[:ENROLLED_IN]->(course:Course)<-[:ENROLLED_IN]-(other:Student)\nWHERE other.student_id <> hong.student_id\nRETURN DISTINCT other.name AS student_name;\n"},
  {'context': [{'student_name': '이민수'}]}]}